# Project 2: word embeddings (Jeff)

GloVe Twitter op de stance data. Stap 1 en 2 van het project.

## Setup

In [ ]:
import sys
from collections import Counter
from pathlib import Path
import pandas as pd
root = Path.cwd().parent
sys.path.insert(0, str(root))
import numpy as np
import part2
from src.data import load_stance_all
from src.embeddings import coverage_table, load_vocab
from src.glove_tokenizer import tokenize, tokenize_plain

GLOVE_PATH = root / "data" / "glove.twitter.27B.50d.txt"  # eerst: uv run python scripts/download_glove.py
GLOVE_DIMS = 50
WIKI_PATH = root / "data" / "glove.6B.50d.txt"  # alternatief: uv run python scripts/download_glove.py wiki

## Data

In [ ]:
stance_dataset = load_stance_all()
train_texts = stance_dataset["train"]["text"]
len(train_texts)

## GloVe Twitter laden

In [ ]:
glove = part2.clslib.PretrainedEmbeddings(str(GLOVE_PATH), GLOVE_DIMS)
len(glove.vocab)

## Stap 1: past het model bij onze data?

GloVe Twitter is getraind op tweets die eerst zijn voorbewerkt (`#feminism` → `<hashtag> feminism`, `@user` → `<user>`, ...).
We doen dezelfde voorbewerking (`src/glove_tokenizer.py`). Alternatief: GloVe 6B (Wikipedia + nieuws).

In [ ]:
example = train_texts[0]
print(example)
print(tokenize(example))

In [ ]:
twitter_vocab = load_vocab(GLOVE_PATH)
wiki_vocab = load_vocab(WIKI_PATH)
naive = lambda text: text.lower().split()

coverage_table(train_texts, {
    "Twitter, split op spaties": (naive, twitter_vocab),
    "Twitter, GloVe-voorbewerking": (tokenize, twitter_vocab),
    "Wikipedia (6B), gewone tokenisatie": (tokenize_plain, wiki_vocab),
})

Met de voorbewerking kent GloVe Twitter 98.8% van de tokens (zonder: 71%). Wikipedia mist vooral Twitter-hashtags (gamergate, lovewins, yesallwomen) die juist de stance dragen.

## Stap 2: wat legt het model vast?

Buren van stance-woorden, vergelijking met een alternatief, beperkingen.

In [ ]:
def cos_similarity(u, v):  
    """
    Return the cosine similarity (a real number) of the two words in the embedding space of our model.
    """      
    dot_product = np.dot(u, v)
    norm_A = np.linalg.norm(u)
    norm_B = np.linalg.norm(v)

    cosine_sim = dot_product / (norm_A * norm_B)
    return(cosine_sim)

def topk_ids(v, E, k=10):    
    """
    word embedding: a D-dimensional vector
    E: our model's embedding matrix as a numpy array of shape [V, D]
    k: how many nearest neighbours we want to find

    Return a list with the k word ids that are nearest to the input word embedding v
     in embedding space according to cosine similarity. 
    """   
    norms_E = np.linalg.norm(E, axis=1)  
    norms_E[norms_E == 0] = 1    
    norm_v = np.linalg.norm(v)

    cosine_sims = (E @ v) / (norms_E * norm_v)  
    return list(np.argsort(-cosine_sims)[:k])

ids = topk_ids(glove.embed_word("feminism"), glove.vectors, k=11)
[glove.word(i) for i in ids[1:]]

In [ ]:
for w in ["person", "human", "place", "city", "car"]:
    print(w, round(float(cos_similarity(glove.embed_word("person"), glove.embed_word(w))), 3))


In [ ]:
wiki = part2.clslib.PretrainedEmbeddings(str(WIKI_PATH), GLOVE_DIMS)

In [ ]:
def neighbours(word, model, k=10):
    """De k woorden die het dichtst bij `word` liggen (het woord zelf niet meegeteld)."""
    if word not in model:
        return ["(onbekend woord)"]
    ids = topk_ids(model.embed_word(word), model.vectors, k=k + 1)
    return [model.word(i) for i in ids[1:]]

In [ ]:
stance_words = ["feminism", "abortion", "prolife", "atheism", "god", "climate", "hillary"]

pd.DataFrame({
    word: {
        "twitter": ", ".join(neighbours(word, glove)),
        "wikipedia": ", ".join(neighbours(word, wiki)),
    }
    for word in stance_words
}).T

Twitter geeft buren uit het online debat (sexism, misogyny, prochoice, obamacare). Wikipedia kent `prolife` nauwelijks: die buren zijn ruis.

### Test 2: liggen tegengestelde standpunten dicht bij elkaar?

Cosinus van paren met een tegengestelde stance vs paren met (bijna) dezelfde betekenis.

In [ ]:
opposite = [("prolife", "prochoice"), ("atheist", "believer"), ("atheist", "christian"), ("feminist", "sexist"),
            ("hillary", "trump"), ("democrat", "republican"), ("liberal", "conservative")]
similar = [("atheist", "atheists"), ("atheist", "nonbeliever"), ("feminist", "feminists"),
           ("hillary", "clinton"), ("climate", "environment"), ("liberal", "progressive")]

def pair_similarity(a, b, model):
    if a not in model or b not in model:
        return np.nan
    return float(cos_similarity(model.embed_word(a), model.embed_word(b)))

pair_table = pd.DataFrame([
    {"paar": f"{a} - {b}", "soort": soort,
     "twitter": pair_similarity(a, b, glove), "wikipedia": pair_similarity(a, b, wiki)}
    for soort, pairs in [("tegengesteld", opposite), ("zelfde betekenis", similar)]
    for a, b in pairs
]).round(2)
pair_table

In [ ]:
pair_table.groupby("soort")[["twitter", "wikipedia"]].mean().round(2)

Tegengestelde standpunten liggen in GloVe Twitter bijna even dicht bij elkaar als woorden met dezelfde betekenis
(gemiddeld 0.74 vs 0.77; prolife - prochoice 0.91 en democrat - republican 0.95 zelfs hoger dan de meeste synoniemen). GloVe leert van woorden die samen voorkomen, en voor- en tegenstanders praten over hetzelfde onderwerp.

### Beperkingen voor onze taak

- Tegengestelde standpunten liggen dicht bij elkaar (test 2): het model weet waar een tweet over gaat, niet welke kant hij kiest.
- Aan elkaar geschreven hashtags ontbreken (prolifeyouth, lovewins, antifeminism), terwijl die vaak de stance dragen (stap 1).
- Eén vector per woord, ongeacht context.
- Getraind op tweets tot 2014: nieuwere woorden en hashtags ontbreken.